# P2 checks — NS10657 (review round, Sept 2026)

Notebook version of the former `p2_checks.py` (converted 2026-09-27).

Run with Python 3.11:

    MPLCONFIGDIR=/tmp/mplcfg /opt/homebrew/bin/python3.11 -m jupyter nbconvert --to notebook --execute 12_p2_checks.ipynb

## Checks
- **C1** — train/validation overlap (released processed CSVs). Asks whether any reported validation used overlapping observations.
- **C4** — constant-anchor-offset check (released processed CSVs).
- **C2** — rejected rows below the `counts_ratio` lower bound (raw scan; requires `RUN_FULL_RAW = True`).
- **C3** — counter/sensor `merge_asof` `|dt|` distribution (raw scan; requires `RUN_FULL_RAW = True`).

C2/C3 need the raw daily CSVs: the public release does not contain the rows rejected by the lower bound, so their distribution cannot be recovered from the released data. Set `RUN_FULL_RAW = True` in the run cell to scan them (slow: reads every daily CSV). Results are written to `p2_checks_results.json` next to this notebook.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Fixed constants of the measurement (copied from
# v1.0/code/data_processing/preprocess_data.ipynb; do not edit here without
# editing PROVENANCE.md as well)
# ---------------------------------------------------------------------------
F_1762_HZ = 170.126432e12        # 1762 nm optical frequency
F_RB_HZ = 384.2281145e12         # rubidium-referenced 780 nm frequency
RATIO_NOMINAL = F_RB_HZ / F_1762_HZ          # = 2.2584857...
COUNTS_RATIO_MIN = 2.2584867
COUNTS_RATIO_MAX = 3.0

# Constant-anchor offset scanned in C4 (units of refractive index).
K_ANCHOR_DEFAULT = 5.2e-6
RESIDUAL_SCATTER = 1.8367e-7     # SM Table S1a residual scatter (sigma_n)

# ---------------------------------------------------------------------------
# Paths — the notebook is expected to run from code/analysis/
# ---------------------------------------------------------------------------
HERE = Path.cwd()
# Locate the v2.0 package: nearest ancestor containing data/processed/.
DEFAULT_V2 = next((p for p in (HERE, *HERE.parents)
                   if (p / "data" / "processed").is_dir()), HERE.parents[1])
DEFAULT_V1 = DEFAULT_V2.parent / "refractive_index_1762nm_v1.0"

V2_PROCESSED = DEFAULT_V2 / "data" / "processed"
# Raw data location (external SSD). RAW_DIR may point either at the raw base
# (the folder holding ``interferometer/`` and ``environmental/``) or directly
# at the ``interferometer/`` folder; the loader always uses RAW_ROOT.
RAW_DIR = Path("/Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw/interferometer")
RAW_ROOT = Path(RAW_DIR).parent if Path(RAW_DIR).name == "interferometer" else Path(RAW_DIR)

# Defensive: tolerate a plain str being assigned above (or in a scratch cell).
RAW_DIR = Path(RAW_DIR)
RAW_ROOT = Path(RAW_ROOT)
V2_PROCESSED = Path(V2_PROCESSED)

# Run control: set True to scan the raw daily CSVs for C2/C3.
RUN_FULL_RAW = True
RAW_START = "20241230"
RAW_END = "20250821"

print("v2 package  :", DEFAULT_V2)
print("v2 processed:", V2_PROCESSED)
print("raw dir     :", RAW_DIR, "(read only when RUN_FULL_RAW)")
print("raw root    :", RAW_ROOT)

v2 package  : /Users/weiwu/github/Ba-Network/zenodo/refractive_index_1762nm_v2.0
v2 processed: /Users/weiwu/github/Ba-Network/zenodo/refractive_index_1762nm_v2.0/data/processed
raw dir     : /Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw/interferometer (read only when RUN_FULL_RAW)
raw root    : /Volumes/PortableSSD/Freiburg/Actomics/barium_network/zenodo/data/raw


## Helpers

Replicated from the v1.0 preprocessor so that C2/C3 reproduce the exact join used to build the release.

In [2]:
def _raw_loader(data_path: Path, start_date: str, end_date: str):
    """Replication of load_interferometer_data() from the v1.0 preprocessor.

    Returns the *unfiltered* per-type concatenated frames (``counts_ratio``,
    ``humidity``, ``temperature``, ``pressure``); filtering and the merge are
    applied by the checks below.
    """
    data_path = Path(data_path)
    # Accept either the raw root (which contains `interferometer/` and
    # `environmental/`) or the `interferometer/` sub-directory itself.
    # This keeps the loader working whether the caller passes RAW_ROOT or
    # RAW_DIR, and whether it passes a str or a Path.
    if data_path.name == "interferometer" and not (data_path / "environmental").is_dir():
        data_path = data_path.parent
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    date_range = [start + timedelta(days=i)
                  for i in range((end - start).days + 1)]

    frames = {"counts_ratio": [], "humidity": [], "temperature": [],
              "pressure": []}
    for date in date_range:
        d = date.strftime("%Y%m%d")
        paths = {
            "counts_ratio": data_path / "interferometer" / f"counts_ratio_data_{d}.csv",
            "humidity": data_path / "environmental" / f"humidity_data_{d}.csv",
            "pressure": data_path / "environmental" / f"pressure_data_{d}.csv",
            "temperature": data_path / "environmental" / f"temperature_data_{d}.csv",
        }
        if not all(p.exists() for p in paths.values()):
            continue
        for kind, path in paths.items():
            df = pd.read_csv(path, encoding="utf-8-sig")
            if "time" in df.columns:
                df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
            frames[kind].append(df)

    raw = {}
    for kind, dfs in frames.items():
        if not dfs:
            raise SystemExit(f"no raw {kind} files found under {data_path}")
        raw[kind] = pd.concat(dfs, ignore_index=True).dropna(
            subset=["time"]).sort_values("time").reset_index(drop=True)

    raw["humidity"] = raw["humidity"].rename(
        columns={"humidity_wm_bme280": "humidity"})
    raw["temperature"] = raw["temperature"].rename(
        columns={"temperature_wm_bme280": "temperature"})
    raw["pressure"] = raw["pressure"].rename(
        columns={"pressure_wm_bme280": "pressure"})
    return raw

In [9]:
def _nearest_dt(left_times: pd.Series, env: pd.DataFrame,
                env_col: str) -> np.ndarray:
    """|dt| (seconds) of the merge_asof(direction='nearest') join."""
    right = env[["time", env_col]].rename(columns={"time": "t_env"})
    left = (left_times.rename("time").to_frame()
            .reset_index(drop=True))
    joined = pd.merge_asof(left, right, left_on="time", right_on="t_env", direction="nearest")
    dt = (joined["t_env"] - joined["time"]).dt.total_seconds()
    return dt.abs().to_numpy()

In [10]:
def _describe(values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=float)
    return {
        "n": int(values.size),
        "median": float(np.median(values)),
        "p95": float(np.percentile(values, 95)),
        "max": float(values.max()),
        "frac_above_10s": float(np.mean(values > 10.0)),
        "frac_above_60s": float(np.mean(values > 60.0)),
    }

## C1 — train/validation overlap

Confirm the 3,459 shared timestamps between `training_data.csv` and `validation_data.csv`, and identify the calendar day(s) responsible.

In [11]:
def check_overlap(v2_processed: Path) -> dict:
    v2_processed = Path(v2_processed)
    tr = pd.read_csv(v2_processed / "training_data.csv",
                     usecols=["time"], parse_dates=["time"], encoding="utf-8-sig")
    va = pd.read_csv(v2_processed / "validation_data.csv",
                     usecols=["time"], parse_dates=["time"], encoding="utf-8-sig")
    fu = pd.read_csv(v2_processed / "full_data.csv",
                     usecols=["time"], parse_dates=["time"], encoding="utf-8-sig")

    tr_t, va_t, fu_t = tr["time"], va["time"], fu["time"]
    n_tr, n_va, n_fu = len(tr_t), len(va_t), len(fu_t)
    shared = set(tr_t) & set(va_t)

    # sanity: full_data should be the union without duplication
    union = set(tr_t) | set(va_t)
    result = {
        "n_training": n_tr,
        "n_validation": n_va,
        "n_full": n_fu,
        "n_training_plus_validation": n_tr + n_va,
        "arithmetic_gap": n_tr + n_va - n_fu,
        "n_shared_timestamps": len(shared),
        "gap_equals_shared": (n_tr + n_va - n_fu) == len(shared),
        "n_union": len(union),
        "union_equals_full": len(union) == n_fu,
        "training_range": [str(tr_t.min()), str(tr_t.max())],
        "validation_range": [str(va_t.min()), str(va_t.max())],
        "full_range": [str(fu_t.min()), str(fu_t.max())],
    }
    if shared:
        s = pd.Series(sorted(shared))
        result["shared_days"] = {str(k): int(v) for k, v in
                                 s.dt.floor("D").value_counts().sort_index().items()}
    return result


results = {}
results["C1_overlap"] = check_overlap(V2_PROCESSED)
for k, v in results["C1_overlap"].items():
    print(f"  {k}: {v}")

  n_training: 108346
  n_validation: 40897
  n_full: 145784
  n_training_plus_validation: 149243
  arithmetic_gap: 3459
  n_shared_timestamps: 3459
  gap_equals_shared: True
  n_union: 145784
  union_equals_full: True
  training_range: ['2025-01-01 00:06:26.245654+00:00', '2025-08-05 19:41:51.652748+00:00']
  validation_range: ['2025-08-05 00:00:06.757350+00:00', '2025-08-20 20:37:37.682980+00:00']
  full_range: ['2025-01-01 00:06:26.245654+00:00', '2025-08-20 20:37:37.682980+00:00']
  shared_days: {'2025-08-05 00:00:00+00:00': 3459}


## C4 — constant-anchor-offset check

`n_1762 = (f_780 / f_1762) * n_780 / R(t)`. A constant offset K on the 780 nm anchor `n_780` appears in `n_1762` as a time-dependent term `K * (f_780/f_1762) / R(t)` whose amplitude is proportional to `K * dR/R`. Quantify it against the residual scatter and the humidity term.

In [12]:
def check_constant_anchor(v2_processed: Path,
                          K: float = K_ANCHOR_DEFAULT) -> dict:
    v2_processed = Path(v2_processed)
    df = pd.read_csv(v2_processed / "full_data.csv", encoding="utf-8-sig")
    R = df["counts_ratio"].to_numpy(dtype=float)
    n1762 = df["n_1762"].to_numpy(dtype=float)

    # invert the measurement equation to recover the 780 nm anchor
    n780 = n1762 * R / RATIO_NOMINAL

    # a constant offset K on the anchor maps to  dn(t) = K * RATIO_NOMINAL / R
    dn = K * RATIO_NOMINAL / R
    dn_mod = dn - dn.mean()

    # residual scatter quoted in the SM is the OLS residual of the full model;
    # here we only need an order-of-magnitude reference, so use the scatter of
    # the series about a simple linear time trend.
    t = np.arange(len(n1762), dtype=float)
    trend = np.polyval(np.polyfit(t, n1762, 1), t)
    scatter = float(np.std(n1762 - trend))

    dr_over_r = float((R.max() - R.min()) / R.mean())
    return {
        "K": float(K),
        "R_mean": float(R.mean()),
        "R_min": float(R.min()),
        "R_max": float(R.max()),
        "dR_over_R_peak_to_peak": dr_over_r,
        "dn_dc_offset": float(dn.mean()),
        "dn_modulation_peak_to_peak": float(dn_mod.max() - dn_mod.min()),
        "dn_modulation_std": float(np.std(dn_mod)),
        "analytic_amplitude_K_times_dR_over_R": float(K * dr_over_r),
        "reference_residual_scatter_sm": RESIDUAL_SCATTER,
        "ratio_modulation_over_residual": float(
            (dn_mod.max() - dn_mod.min()) / RESIDUAL_SCATTER),
        "scatter_detrended_linear": scatter,
        "verdict": ("PASS - the constant-anchor term is many orders of "
                    "magnitude below the residual scatter"
                    if (dn_mod.max() - dn_mod.min()) < 1e-3 * RESIDUAL_SCATTER
                    else "REVIEW"),
    }


results["C4_constant_anchor"] = check_constant_anchor(V2_PROCESSED, K=K_ANCHOR_DEFAULT)
for k, v in results["C4_constant_anchor"].items():
    print(f"  {k}: {v}")

  K: 5.2e-06
  R_mean: 2.2584875217650766
  R_min: 2.2584867
  R_max: 2.2584884
  dR_over_R_peak_to_peak: 7.527161356879003e-07
  dn_dc_offset: 5.199995814128303e-06
  dn_modulation_peak_to_peak: 3.91412065688893e-12
  dn_modulation_std: 9.953620844964293e-13
  analytic_amplitude_K_times_dR_over_R: 3.914123905577082e-12
  reference_residual_scatter_sm: 1.8367e-07
  ratio_modulation_over_residual: 2.1310614999123043e-05
  scatter_detrended_linear: 2.8055443445032307e-06
  verdict: PASS - the constant-anchor term is many orders of magnitude below the residual scatter


## C2 / C3 — raw scan (optional)

**Requires `RUN_FULL_RAW = True`** (set it in the run cell below). The raw directory is set by `RAW_DIR` (external SSD path); this scans every daily CSV.

- **C2** — rows rejected by `counts_ratio in [2.2584867, 3]`, and whether the rejection preferentially removes humid rows.
- **C3** — distribution of `|dt|` for the `merge_asof(direction='nearest')` join (the counter/sensor matching distance).

In [13]:
def check_raw(raw_dir: Path, start_date: str, end_date: str,
              keep_frames: bool = True) -> dict:
    raw_dir = Path(raw_dir)
    raw = _raw_loader(raw_dir, start_date, end_date)
    cr = raw["counts_ratio"]
    mask = ((cr["counts_ratio"] >= COUNTS_RATIO_MIN) &
            (cr["counts_ratio"] <= COUNTS_RATIO_MAX))
    kept, dropped = cr[mask], cr[~mask]

    out = {
        "n_counts_rows_total": int(len(cr)),
        "n_kept": int(len(kept)),
        "n_dropped": int(len(dropped)),
        "kept_fraction": float(len(kept) / len(cr)),
        "counts_ratio_kept_min": float(kept["counts_ratio"].min()),
        "counts_ratio_kept_max": float(kept["counts_ratio"].max()),
        "counts_ratio_span_rel": float(
            (kept["counts_ratio"].max() - kept["counts_ratio"].min()) /
            RATIO_NOMINAL),
        "counts_ratio_all_min": float(cr["counts_ratio"].min()),
        "counts_ratio_all_max": float(cr["counts_ratio"].max()),
    }
    if len(dropped):
        out["dropped_ratio_hist"] = {
            "below_min": int((dropped["counts_ratio"] < COUNTS_RATIO_MIN).sum()),
            "above_max": int((dropped["counts_ratio"] > COUNTS_RATIO_MAX).sum()),
            "below_min_values": [float(v) for v in
                                 dropped.loc[dropped["counts_ratio"] < COUNTS_RATIO_MIN,
                                             "counts_ratio"].head(20)],
        }
    out["dropped_by_month"] = (
        dropped.assign(m=dropped["time"].dt.tz_localize(None).dt.to_period("M").astype(str))
        .groupby("m").size().to_dict() if len(dropped) else {})

    # Attach the environmental channels exactly as the preprocessor does, so
    # that the T/H/P comparison and the |dt| statistics are on the same rows.
    merged = kept
    dt_stats = {}
    for env_col in ("humidity", "temperature", "pressure"):
        dt_stats[env_col] = _describe(
            _nearest_dt(merged["time"], raw[env_col], env_col))
        merged = pd.merge_asof(merged, raw[env_col][["time", env_col]],
                               on="time", direction="nearest")

    dropped_env = dropped
    for env_col in ("humidity", "temperature", "pressure"):
        dropped_env = pd.merge_asof(dropped_env,
                                    raw[env_col][["time", env_col]],
                                    on="time", direction="nearest")

    def _range(df, col):
        v = df[col].dropna()
        return [float(v.min()), float(np.median(v)), float(v.max())]

    out["nearest_neighbour_dt_seconds"] = dt_stats
    out["env_ranges"] = {
        "kept": {c: _range(merged, c)
                 for c in ("temperature", "humidity", "pressure")},
        "dropped": {c: _range(dropped_env, c)
                    for c in ("temperature", "humidity", "pressure")},
    }
    out["env_ranges_legend"] = "[min, median, max]"

    # Does the rejection preferentially remove humid rows?
    if len(dropped) and len(merged):
        hi_kept = float((merged["humidity"] > 80).mean())
        hi_drop = float((dropped_env["humidity"] > 80).mean())
        out["fraction_rh_above_80pct"] = {"kept": hi_kept, "dropped": hi_drop}
    return out

In [14]:
# Set RUN_FULL_RAW = True (cell above) to scan the raw daily CSVs for C2/C3.
if RUN_FULL_RAW:
    out = check_raw(Path(RAW_ROOT), RAW_START, RAW_END)
    results["C2_rejected_rows"] = {
        k: v for k, v in out.items()
        if k in ("n_counts_rows_total", "n_kept", "n_dropped",
                 "kept_fraction", "counts_ratio_kept_min",
                 "counts_ratio_kept_max", "counts_ratio_span_rel",
                 "counts_ratio_all_min", "counts_ratio_all_max",
                 "dropped_ratio_hist", "dropped_by_month",
                 "fraction_rh_above_80pct")}
    results["C3_time_mismatch"] = {
        "nearest_neighbour_dt_seconds": out["nearest_neighbour_dt_seconds"],
        "env_ranges": out["env_ranges"],
        "env_ranges_legend": out["env_ranges_legend"]}
    for section in ("C2_rejected_rows", "C3_time_mismatch"):
        print(f"-- {section} --")
        for k, v in results[section].items():
            print(f"  {k}: {v}")
        print()
else:
    print("C2/C3 skipped: set RUN_FULL_RAW = True to scan the raw daily CSVs "
          "for the rejected-row statistics and the counter/sensor time mismatch.")

-- C2_rejected_rows --
  n_counts_rows_total: 156931
  n_kept: 145784
  n_dropped: 11147
  kept_fraction: 0.9289687824585328
  counts_ratio_kept_min: 2.2584867
  counts_ratio_kept_max: 2.2584884
  counts_ratio_span_rel: 7.527167416063325e-07
  counts_ratio_all_min: 2.2584858
  counts_ratio_all_max: 2.2584884
  dropped_ratio_hist: {'below_min': 11147, 'above_max': 0, 'below_min_values': [2.2584859, 2.2584863, 2.258486, 2.2584862, 2.2584862, 2.2584866, 2.2584858, 2.2584861, 2.2584862, 2.2584859, 2.2584858, 2.2584865, 2.2584858, 2.2584859, 2.2584859, 2.2584859, 2.2584861, 2.258486, 2.2584866, 2.2584859]}
  dropped_by_month: {'2025-01': 2001, '2025-02': 11, '2025-03': 1, '2025-06': 256, '2025-07': 2640, '2025-08': 6238}
  fraction_rh_above_80pct: {'kept': 0.0, 'dropped': 0.0}

-- C3_time_mismatch --
  nearest_neighbour_dt_seconds: {'humidity': {'n': 145784, 'median': 0.406949, 'p95': 4.2601736500000005, 'max': 458.468499, 'frac_above_10s': 0.014692970421994184, 'frac_above_60s': 0.00013032

In [15]:
results["generated"] = datetime.utcnow().isoformat() + "Z"
results["v2_processed"] = str(V2_PROCESSED)
results["raw_dir"] = str(RAW_DIR)
results["raw_root"] = str(RAW_ROOT)

out_path = HERE / "p2_checks_results.json"
with open(out_path, "w") as fh:
    json.dump(results, fh, indent=2)
print("written:", out_path)

written: /Users/weiwu/github/Ba-Network/zenodo/refractive_index_1762nm_v2.0/code/analysis/p2_checks_results.json
